# Washington (WSCJTC) 2026

`PRR_7.29.26.xlsx`, sheet `ALL`: one row per officer with up to 12 employment stints side by side. This reshapes it to one row per stint; cleaning starts below.

In [4]:
import pandas as pd

In [5]:
# def read_tbl(input_dir="../data/input"):
#     """Read the ALL sheet: one row per officer, employment stints side by
#     side."""
#     return pd.read_excel(
#         os.path.join(input_dir, "PRR_7.29.26.xlsx"),
#         sheet_name="ALL",
#         dtype=str,
#     )


# def snake_case(name):
#     name = re.sub(r"\(current\)", "", name.lower())
#     return re.sub(r"[^a-z0-9]+", "_", name).strip("_")


# def employment_groups(columns):
#     """Map each stint's columns to their base name, one dict per stint.

#     A stint starts at each "Employment Start Date" column; the export drops
#     columns empty for every officer, so later stints have fewer fields.
#     """
#     groups = []
#     for col in columns:
#         base = re.sub(r"\.\d+$", "", col)
#         if base == "Employment Start Date":
#             groups.append({})
#         # agency addresses and certification columns are not stint fields
#         if groups and base.startswith(("Employment ", "Employing ")):
#             groups[-1].setdefault(base, col)
#     return groups


# def melt_stints(df):
#     """Reshape to one row per officer per stint, dropping empty stints."""
#     person_cols = [c for c in df.columns if c.startswith("Person ")]
#     groups = employment_groups(df.columns)
#     fields = list(dict.fromkeys(base for g in groups for base in g))

#     frames = []
#     for i, group in enumerate(groups, start=1):
#         stint = df[person_cols].copy()
#         for base in fields:
#             stint.loc[:, base] = df[group[base]] if base in group else None
#         stint.loc[:, "stint_nbr"] = i
#         frames.append(stint)
#     out = pd.concat(frames, ignore_index=True)

#     out = out.dropna(subset=fields, how="all")
#     out.columns = [snake_case(c) for c in out.columns]
#     return out.sort_values(["person_student_id", "stint_nbr"]).reset_index(
#         drop=True
#     )

# df_all = read_tbl()
# df = melt_stints(df_all)


# print(f"{len(df_all):,} officers → {len(df):,} stints")
# df.head()

## Cleaning

In [6]:
df = pd.read_csv("../data/input/melted_stints.csv")

df.head(20)


def rename_cols(df):
    df = df.rename(
        columns={
            "person_student_id": "person_nbr",
            "person_first_name": "first_name",
            "person_middle_name": "middle_name",
            "person_last_name": "last_name",
            "person_suffix": "suffix",
            "person_gender": "sex",
            "employment_start_date": "start_date",
            "employment_end_date": "end_date",
            "employment_appointment_type": "certification_type",  # should this be called certification type? add it to the readme with some examples @claudina
            "employment_title_rank": "rank",
            "employment_status": "status",
            "employment_change_reason": "separation_reason",
            "employing_organization_name": "agency_name",
            "employing_organization_ori_or_wa_ubi": "ori",
        }
    )
    return df.drop(
        columns=[
            "ori",
            "stint_nbr",
            "employing_organization_is_employer",
            "employment_employment_assignment",
        ]
    )


def clean_dates(df):
    df.loc[:, "start_date"] = (
        df.start_date.fillna("")
        .astype(str)
        .str.replace(r"\-(\w{2}) (.+)", r"-\1", regex=True)
    )
    df.loc[:, "end_date"] = (
        df.end_date.fillna("")
        .astype(str)
        .str.replace(r"-(\w{2}) (.+)", r"-\1", regex=True)
    )

    return df


def clean_names(df):
    df.loc[:, "first_name"] = df.first_name.fillna("")
    df.loc[:, "middle_name"] = df.middle_name.fillna("")
    df.loc[:, "last_name"] = df.last_name.fillna("")
    df.loc[:, "suffix"] = df.suffix.fillna("")
    return df


df = df.pipe(rename_cols).pipe(clean_dates).pipe(clean_names)

df

# we need to clean the other columns @claudina
# we might want to drop everyone who isn't a certified peace officer  @claudina. we should ask the team. remind me immediately

df.agency_name.unique().tolist()

['NAS Whidbey Island Police',
 'Inter-Con Security Services, Inc.',
 'North East King County Regional Communications',
 'Allied Universal Security Services',
 'Universal Protection Service, LP',
 'Liberty Security Patrol',
 'Lummi Nation Police Department',
 'La Conner Police Department',
 'Puget Sound Executive Services',
 'Bellevue Police Department',
 'Sultan Police Department',
 'King County Department of Adult and Juvenile Detention',
 'WA State Patrol (WSP)',
 'Kent Police Department',
 'Tri-County Juvenile Department (Ferry, Pend Oreille, Stevens)',
 'Bluetac Protection Services',
 'Pacific River, LLC',
 'Executive Security Services, Inc.',
 'Off Duty Services, Inc.',
 'Kraken Protective Services',
 'United States Army',
 'Monroe Correctional Complex',
 'Kerberos International',
 "King County Sheriff's Office",
 'Seattle Police Department',
 "Island County Sheriff's Office",
 'Mount Vernon Police Department',
 'Basic Contracting Services, Inc.',
 'Phoenix Protective Corp',
 'Pie